# Interpreting ADA


| section | run it |
|---|---|
| **1. Imports & paths** | once |
| **2. Function library** | once |
| **3. Model & artifacts** | once per model |
| **4. Display options** | edit + rerun freely — no recomputation |
| **5. Compare the three methods** | the output |

Three methods to interpretate the accentuated directions, all for one support set at one layer:

1. **`d`** — the context displacement vector, `h(target | list) − h(target | alone) − mean_disp`. What context actually did.
2. **`A @ p`** — the predicted displacement using the fitted ridge map (A) & the prototype of the support set.
3. **Prototype-aligned singular vectors of `A`** — which axes this support set excites, read
   out on the input side (`u_i`, what points along the axis), the output side (`v_i`, what the axis
   writes), or both.

**Prerequisite:** `latent_regression.py` might need to be rerun since it was updated to save `all_A.npy`

```bash
cd few_shot/src && python latent_regression.py --model_name google/gemma-2-2b
```

**On E vs U:** when a model ties its input and output embeddings (Gemma-2 does), `E` and `U` are the
same matrix and only one column is printed. Untied models (OLMo, Llama) get both.

---
## 1. Imports & paths

In [11]:
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

# Repo layout: <root>/notebooks/<this file>, <root>/few_shot/src/, <root>/few_shot/results/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd()))
SRC_DIR   = os.path.join(REPO_ROOT, "..", "src")
sys.path.insert(0, SRC_DIR)

from helpers import data as data_module
from helpers import model_helpers

DATA_PATH = os.path.join(REPO_ROOT, "..", "data", "categories_data.csv")
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 250)

print("repo root", REPO_ROOT)
print("device   ", DEVICE)

repo root /oscar/data/epavlick/mlepori/projects/rerepresentation/few_shot/analysis
device    cpu


---
## 2. Function library

### 2a. Loading

In [5]:
def load_model_and_matrices(model_name, device, readout_device=None):
    """Load the model plus both embedding/unembedding matrices, deduplicated when the model ties them.

    Returns a dict with everything the readouts need.  The (vocab x d) matrices are ~2.4 GB
    each in fp32 for Gemma-2-2b, so a tied model materialises only one copy; pass
    readout_device="cpu" if the GPU is tight (the readouts are single matvecs either way).
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer

    readout_device = readout_device or device

    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name, dtype="float32", trust_remote_code=True,   # `torch_dtype=` also works
    )
    mdl.eval().to(device)

    emb_w = mdl.get_input_embeddings().weight
    head  = mdl.get_output_embeddings()
    unw   = head.weight if head is not None else emb_w
    tied  = emb_w.data_ptr() == unw.data_ptr()

    with torch.no_grad():
        emb_t = emb_w.detach().float().to(readout_device)
        emb_n = emb_t / emb_t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        if tied:
            unemb_t, unemb_n = emb_t, emb_n
        else:
            unemb_t = unw.detach().float().to(readout_device)
            unemb_n = unemb_t / unemb_t.norm(dim=-1, keepdim=True).clamp(min=1e-12)

    final_norm, norm_path = _find_final_norm(mdl)

    return {
        "model": mdl, "tokenizer": tok, "tied": tied,
        "EMB_T": emb_t, "EMB_N": emb_n, "UNEMB_T": unemb_t, "UNEMB_N": unemb_n,
        "final_norm": final_norm, "norm_path": norm_path,
        "softcap": getattr(mdl.config, "final_logit_softcapping", None),
        "context_id": tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id,
        "d_model": emb_t.shape[1],
        "readout_device": readout_device,
    }


def _find_final_norm(m):
    """Locate the final norm module; layout varies across transformers versions."""
    for path in ("model.norm", "model.language_model.norm", "transformer.ln_f", "model.final_layernorm"):
        obj = m
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
            return obj, path
        except AttributeError:
            continue
    raise AttributeError("Could not locate the final norm; set FINAL_NORM by hand.")


def load_artifacts(results_dir_concept, results_dir_regression, model_name, src_dir):
    """Load artifacts with a clear error if it is stale."""
    def _one_concept(name, results_dir_concept, **kw):
        path = os.path.join(results_dir_concept, name)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found.\n"
                f"Rerun:  cd {src_dir} && python harvest_model_data.py --model_name {model_name}"
            )
        if name.endswith(".npy"):
            return np.load(path, **kw)
        if name.endswith(".csv"):
            return pd.read_csv(path)
        if name.endswith(".json"):
            return json.load(open(path))

    def _one_regression(name, results_dir_regression, **kw):
        path = os.path.join(results_dir_regression, name)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found.\n"
                f"Rerun:  cd {src_dir} && python latent_regression.py --model_name {model_name}"
            )
        return np.load(path, **kw) if name.endswith(".npy") else json.load(open(path))

    return {
        "ridge_maps": _one_regression("all_A.npy", results_dir_regression, mmap_mode="r"),   # (n_layers, d, d)
        "hparams":    _one_regression("hyperparams.json", results_dir_regression),
        "mean_disp":  _one_concept("global_mean_displacement.npy", results_dir_concept),           # (n_layers, d)
        "records":    _one_concept("records.csv", results_dir_concept),
     
    }

### 2b. Readouts

`cos` is the raw cosine against the readout matrix — "what points along this direction".
`lens` is the real logit lens (final RMSNorm → matrix → softcap → vocab-centre) — "what this
direction promotes". They differ by the norm's learned per-dimension gain `(1 + w)`, a diagonal
reweighting that genuinely reorders the token list, so disagreement between them is informative.

In [6]:
def _top_bottom_tokens(scores, n):
    """Return (top_tokens, bottom_tokens) for a (vocab,) score array. Tokens only, no values."""
    order = np.argsort(scores)
    return (
        tokenizer.convert_ids_to_tokens(order[::-1][:n].tolist()),
        tokenizer.convert_ids_to_tokens(order[:n].tolist()),
    )


def cosine_readout(vec, which="E", n=None):
    """Cosine similarity between `vec` and every row of E (or U). Returns (top, bottom) tokens."""
    n = n or N_TOP_TOKENS
    v = torch.as_tensor(np.asarray(vec), dtype=torch.float32, device=READOUT_DEVICE)
    v = v / v.norm().clamp(min=1e-12)
    M = EMB_N if which == "E" else UNEMB_N
    with torch.no_grad():
        sims = (M @ v).float().cpu().numpy()
    return _top_bottom_tokens(sims, n)


def lens_readout(vec, which="E", n=None, apply_norm=True, center=None):
    """Logit lens: final RMSNorm -> readout matrix -> softcap -> vocab-centre."""
    n = n or N_TOP_TOKENS
    center = CENTER_LOGITS if center is None else center
    x = torch.as_tensor(np.asarray(vec), dtype=torch.float32, device=DEVICE).view(1, -1)
    W = EMB_T if which == "E" else UNEMB_T
    with torch.no_grad():
        if apply_norm:
            x = FINAL_NORM(x)
        logits = (x.float().to(READOUT_DEVICE) @ W.T).squeeze(0)
        if SOFTCAP:                                    # monotonic; does not reorder top-k
            logits = SOFTCAP * torch.tanh(logits / SOFTCAP)
        if center:
            logits = logits - logits.mean()
    return _top_bottom_tokens(logits.float().cpu().numpy(), n)


def active_readouts(show=None):
    """Resolve SHOW into [(label, fn), ...].

    SHOW is "cos" | "lens" | "both".  When the model ties E and U there is only one matrix,
    so labels drop the suffix and a single column is produced per kind.
    """
    show = (SHOW if show is None else show).lower()
    kinds = [k for k in (("cos", cosine_readout), ("lens", lens_readout))
             if show in (k[0], "both")]
    if not kinds:
        raise ValueError(f"SHOW must be 'cos', 'lens', or 'both'; got {show!r}")

    out = []
    for kname, kfn in kinds:
        if TIED:
            out.append((kname, lambda v, n=None, f=kfn: f(v, "E", n=n)))
        else:
            for m in ("E", "U"):
                out.append((f"{kname}-{m}", lambda v, n=None, f=kfn, m=m: f(v, m, n=n)))
    return out

### 2c. Display

In [7]:
def compare_directions(named_vecs, n=None, which="top", show=None):
    """Side-by-side table of top (or bottom) tokens.

    Columns are a (direction, readout) MultiIndex; rows are ranks. Tokens only, no scores.
    """
    n  = n or N_TOP_TOKENS
    ro = active_readouts(show)

    cols = {}
    for dname, vec in named_vecs.items():
        for rname, fn in ro:
            top_t, bot_t = fn(vec, n=n)
            toks = (top_t if which == "top" else bot_t)[:n]
            cols[(dname, rname)] = list(toks) + [""] * (n - len(toks))

    df = pd.DataFrame(cols)
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["direction", "readout"])
    df.index   = [f"#{i + 1}" for i in range(n)]
    return df


def print_directions(named_vecs, n=None, which="top", show=None):
    """Compact one-line-per-readout print. Tokens only."""
    n = n or N_TOP_TOKENS
    for dname, vec in named_vecs.items():
        print(f"\n{dname}")
        for rname, fn in active_readouts(show):
            top_t, bot_t = fn(vec, n=n)
            toks = (top_t if which == "top" else bot_t)[:n]
            print(f"    {rname:<8} {' '.join(toks)}")

### 2d. The three methods

In [18]:
def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


_SVD_CACHE = {}   # (layer, solver) -> (input axes as rows, sigma, output axes as rows)


def layer_svd(layer, k=None, solver=None):
    """SVD of the ridge map A^layer, computed from all_A on demand and cached.

    The map is applied as d = p @ A, so with A = U S Vt:
        p @ A = sum_i sigma_i (p . u_i) v_i,   u_i = U[:, i],  v_i = Vt[i, :]
    i.e. the INPUT axis u_i (prototype space) is paired with the OUTPUT axis v_i
    (displacement space) at the same index -- which is why the pair has to be
    sign-flipped together downstream.

    Only the input side was ever written to disk (top_input_svecs.npy), so both
    sides are recomputed here rather than loaded.

    SVD_SOLVER:
        "full"       exact np.linalg.svd, the whole spectrum. Minutes per layer.
        "randomized" sklearn randomized_svd, top-k triplets only. Seconds per layer.

    Returns (in_svecs (k, d), sigma (k,), out_svecs (k, d)).
    """
    k      = k or K_SVD_COMPONENTS
    solver = (solver or SVD_SOLVER).lower()

    cached = _SVD_CACHE.get((layer, solver))
    if cached is None or cached[1].shape[0] < k:      # miss, or cached too few components
        A = np.asarray(RIDGE_MAPS[layer], dtype=np.float32)   # memory-mapped: slice one layer
        print(f"[svd] layer {layer}: {solver} SVD of A {A.shape} ...", flush=True)
        if solver == "full":
            U, sigma, Vt = np.linalg.svd(A, full_matrices=False)
        elif solver == "randomized":
            from sklearn.utils.extmath import randomized_svd
            U, sigma, Vt = randomized_svd(A, n_components=k, n_oversamples=20,
                                          n_iter=7, random_state=SEED)
        else:
            raise ValueError(f"SVD_SOLVER must be 'full' or 'randomized'; got {solver!r}")
        cached = _SVD_CACHE[(layer, solver)] = (np.ascontiguousarray(U.T), sigma, Vt)
        print(f"[svd] layer {layer}: cached {sigma.shape[0]} components", flush=True)

    in_rows, sigma, out_rows = cached
    return in_rows[:k], sigma[:k], out_rows[:k]


def pick_support_set(category=None, tokens=None, which=0):
    """Return (tokens, provenance). Explicit `tokens` wins; otherwise prefer a HELD-OUT record."""
    if tokens is not None:
        bad = [t for t in tokens if len(tokenizer.encode(" " + t, add_special_tokens=False)) != 1]
        if bad:
            print(f"WARNING: not single-token with a leading space: {bad}. Everything still "
                  f"computes, but this support set is no longer comparable to the fitted run.")
        return list(tokens), "user-specified"

    if category is None:
        raise ValueError("Set either CATEGORY or TOKENS.")
    if category not in ALL_CATS:
        close = [c for c in ALL_CATS if category.lower() in c.lower()][:10]
        raise ValueError(f"'{category}' not in the data. Did you mean: {close or ALL_CATS[:10]}")

    heldout = RECORDS[(RECORDS["category"] == category) & (RECORDS["split"] == "heldout")]
    if len(heldout):
        rec = heldout.iloc[which % len(heldout)]
        return (str(rec["context_tokens"]).split(),
                f"held-out record (support_id={rec['support_id']}) from the fitted run")

    members = FILTERED_DATA[FILTERED_DATA["category"] == category]["category.member"].tolist()
    rng = np.random.default_rng(SEED + which)
    support_len = int(RECORDS["context_tokens"].str.split().str.len().mode().iat[0])
    return (rng.choice(members, size=support_len, replace=False).tolist(),
            "freshly sampled (no held-out record for this category)")


def compute_support(tokens):
    """Prototype + displacement for one support set. Returns a dict of (n_layers, d) arrays."""
    target = tokens[-1]                       # last token is the one whose representation we track
    with torch.no_grad():
        _hs, _ = model_helpers.get_hidden_states_and_logprobs_last_token(
            " " + " ".join(tokens), tokenizer, model, CONTEXT_ID)
        h_context = _hs.numpy()
        isolated = {t: model_helpers.get_hidden_states_and_logprobs_last_token(
            " " + t, tokenizer, model, CONTEXT_ID)[0].numpy() for t in set(tokens)}

    h_alone   = isolated[target]
    prototype = np.stack([isolated[t][0] for t in tokens]).mean(0)      # layer-0 mean

    assert h_context.shape == MEAN_DISP.shape, (
        f"layer/dim mismatch: hidden states {h_context.shape} vs saved mean_disp "
        f"{MEAN_DISP.shape}. The artifacts came from a different model — check MODEL_NAME."
    )

    return {
        "tokens": tokens, "target": target, "prototype": prototype,
        "h_context": h_context, "h_alone": h_alone,
        "disp_raw": h_context - h_alone,
        "disp": (h_context - h_alone) - MEAN_DISP,     # METHOD 1
    }


def predicted_displacement(p, layer):
    """METHOD 2: d_pred = p @ A^layer. RIDGE_MAPS is memory-mapped, so slice one layer."""
    A = np.asarray(RIDGE_MAPS[layer], dtype=np.float32)
    return p.astype(np.float32) @ A


def align_spectrum(p, layer):
    """METHOD 3: rank the saved singular axes of A^layer by alignment with prototype p.

    abs_cos : how much of the prototype points along this axis
    gain    : sigma * |cos| — how much the axis contributes to p @ A
    energy  : (sigma * cos)^2, normalised over the axes layer_svd returned (top
              K_SVD_COMPONENTS, which is the full rank of A only when SVD_SOLVER is
              "full" and K_SVD_COMPONENTS >= d), so read it as a share within that subspace
    """
    U, sig, _ = layer_svd(layer)
    p_hat = p / np.linalg.norm(p)
    U_hat = U / np.linalg.norm(U, axis=1, keepdims=True).clip(1e-12)

    c      = U_hat @ p_hat                       # signed cosine
    energy = (sig * c) ** 2
    energy = energy / energy.sum() if energy.sum() > 0 else energy

    return pd.DataFrame({
        "sv": np.arange(len(c)), "sigma": sig, "cos": c,
        "abs_cos": np.abs(c), "gain": sig * np.abs(c), "energy": energy,
    })


def sv_directions(p, layer, k=None, rank_by=None, side=None):
    """Top-k prototype-aligned axes as {label: vector}, oriented toward the prototype.

    side is "input" | "output" | "both".  A negative cos means the prototype points along
    -u_i, which is stretched by the same sigma, so vectors are sign-flipped to face the
    prototype and "top tokens" always means the end of the axis this support set excites.
    """
    k       = k or K_SVECS
    rank_by = rank_by or RANK_BY
    side    = (side or SV_SIDE).lower()
    if side not in ("input", "output", "both"):
        raise ValueError(f"SV_SIDE must be 'input', 'output', or 'both'; got {side!r}")

    in_svecs, _, out_svecs = layer_svd(layer)
    top = align_spectrum(p, layer).sort_values(rank_by, ascending=False).head(k)
    out = {}
    for _, row in top.iterrows():
        i    = int(row["sv"])
        sign = np.sign(row["cos"]) or 1.0
        if side in ("input", "both"):
            out[f"SV{i} in"]  = in_svecs[i]  * sign
        if side in ("output", "both"):
            out[f"SV{i} out"] = out_svecs[i] * sign
    return out, top

---
## 3. Model & artifacts

Run once per model. Everything downstream reads the globals set here.

In [12]:
MODEL_NAME     = "google/gemma-2-2b"
READOUT_DEVICE = DEVICE       # set to "cpu" if the GPU is tight
SEED           = 19           # matches latent_regression.py

MODEL_ID    = MODEL_NAME.replace("/", "_")
RESULTS_DIR_regression = os.path.join(REPO_ROOT, "..", "results_final/categories", MODEL_ID, "displacement_regression")
RESULTS_DIR_concept = os.path.join(REPO_ROOT, "..", "results_final/categories", MODEL_ID, "concept_list_data")
# ── model ────────────────────────────────────────────────────────────────────
_M = load_model_and_matrices(MODEL_NAME, DEVICE, READOUT_DEVICE)
model, tokenizer = _M["model"], _M["tokenizer"]
TIED             = _M["tied"]
EMB_T, EMB_N     = _M["EMB_T"], _M["EMB_N"]
UNEMB_T, UNEMB_N = _M["UNEMB_T"], _M["UNEMB_N"]
FINAL_NORM       = _M["final_norm"]
SOFTCAP          = _M["softcap"]
CONTEXT_ID       = _M["context_id"]
D_MODEL          = _M["d_model"]

print(f"{MODEL_NAME}  |  d_model {D_MODEL}  |  tied E/U: {TIED}")
print(f"final norm = model.{_M['norm_path']}  |  logit softcap = {SOFTCAP}")
if TIED:
    print("  -> E and U are the same matrix; readouts print ONE column per kind.")
else:
    print("  -> E and U differ; readouts print separate -E and -U columns.")

# ── artifacts ────────────────────────────────────────────────────────────────
_A = load_artifacts(RESULTS_DIR_concept, RESULTS_DIR_regression, MODEL_NAME, SRC_DIR)
RIDGE_MAPS = _A["ridge_maps"]
MEAN_DISP  = _A["mean_disp"]
RECORDS    = _A["records"]
HPARAMS    = _A["hparams"]
N_LAYERS   = RIDGE_MAPS.shape[0]

print(f"\nridge_maps {RIDGE_MAPS.shape}   (singular vectors are recomputed per layer "
      f"from all_A by layer_svd, on first use)")
print(f"n_layers   {N_LAYERS}  (index 0 = embeddings, index L = after block L)")

# ── category data ────────────────────────────────────────────────────────────
FILTERED_DATA = data_module.filter_multitoken_data(pd.read_csv(DATA_PATH), tokenizer)
ALL_CATS      = sorted(FILTERED_DATA["category"].unique())
print(f"{len(FILTERED_DATA)} single-token members across {len(ALL_CATS)} categories")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

google/gemma-2-2b  |  d_model 2304  |  tied E/U: True
final norm = model.model.norm  |  logit softcap = 30.0
  -> E and U are the same matrix; readouts print ONE column per kind.

ridge_maps (27, 2304, 2304)   (singular vectors are recomputed per layer from all_A by layer_svd, on first use)
n_layers   27  (index 0 = embeddings, index L = after block L)
3242 single-token members across 113 categories


---
## 4. Display/compute options

Cheap to rerun — nothing here recomputes hidden states or touches the model weights.

| knob | values |
|---|---|
| `SHOW` | Show decoded top tokens computed from raw cosine similarities, or after passing through the normailization layer as in other logit lens methods
| `RANK_BY` | How to rank the SVs  |
| `SV_SIDE` | `"input"` (what points along the axis) \| `"output"` (what it writes) \| `"both"` |
| `SVD_SOLVER` | `"randomized"` (top-`K_SVD_COMPONENTS` triplets, seconds per layer) \| `"full"` (exact whole spectrum, minutes per layer) |
| `K_SVD_COMPONENTS` | How many top SVs to consider if randomized was chosen in the solver |

The singular vectors are recomputed from `all_A` rather than read from
`top_input_svecs.npy`, so the input and output sides come from one factorisation. The first
call for a layer pays the SVD; after that it is cached in `_SVD_CACHE`, keyed by
`(layer, SVD_SOLVER)`. Changing `SVD_SOLVER` or raising `K_SVD_COMPONENTS` above what is
cached triggers a recompute; lowering it does not.

In [13]:
SHOW          = "cos"     # "cos" | "lens" | "both"
RANK_BY       = "abs_cos"     # "gain" | "abs_cos" | "energy" | "sigma" for SVs
SV_SIDE       = "both"     # "input" | "output" | "both"

SVD_SOLVER       = "randomized"   # "randomized" (fast, top-k) | "full" (exact, whole spectrum)
K_SVD_COMPONENTS = 100            # singular triplets kept per layer

N_TOP_TOKENS  = 10         # tokens per readout
K_SVECS       = 5          # how many prototype-aligned axes to inspect
CENTER_LOGITS = True       # subtract the vocab mean from lens logits (kills the frequency baseline)
SHOW_BOTTOM   = True      # also show the most-suppressed tokens
SWEEP_LAYERS  = [6, 10, 14, 16, 18, 20, 21, 22, 23, 24]   # used by the sweep and control cells

print(f"readouts: {[name for name, _ in active_readouts()]}")
print(f"axes:     top {K_SVECS} by {RANK_BY}, side={SV_SIDE}")
print(f"svd:      {SVD_SOLVER}, {K_SVD_COMPONENTS} components/layer, "
      f"{len(_SVD_CACHE)} layer(s) cached")

readouts: ['cos']
axes:     top 5 by abs_cos, side=both
svd:      randomized, 100 components/layer, 0 layer(s) cached


---
## 5. Compare the three methods

### 5a. Sweep layers

Same support set, the layer varying.
Edit and rerun this to change what you are looking at. \
When `CATEGORY` is specified, a **held-out** support set is pulled from 
from `records.json` (so the prototype never touched the ridge fit); \
When `TOKENS` is specified, it overrides CATEGORY.

In [ ]:
SWEEP_LAYERS = [21, 22, 23]
CATEGORY     = "bird"
SUPPORT_PICK = 0
TOKENS       = None

support_tokens, provenance = pick_support_set(CATEGORY, TOKENS, SUPPORT_PICK)
S = compute_support(support_tokens)

print(f"{'=' * 110}\nPROTOTYPE p   {support_tokens}   (layer-independent)\n{'=' * 110}")
print_directions({"p": S["prototype"]})

for L in SWEEP_LAYERS:
    ap = predicted_displacement(S["prototype"], L)
    print(f"\n{'=' * 110}\nLAYER {L}   cos(A@p, d) = {cos(ap, S['disp'][L]):+.3f}\n{'=' * 110}")
    print_directions({"d": S["disp"][L], "A@p": ap})

    svs, tbl = sv_directions(S["prototype"], L)
    print("")
    print(f"aligned axes (ranked by {RANK_BY}): {list(tbl['sv'].astype(int))}")
    abs_cos_map = dict(zip(tbl["sv"].astype(int), tbl["abs_cos"]))
    for label, vec in svs.items():
        sv_i = int(label.split("SV")[1].split()[0])
        ac   = abs_cos_map[sv_i]
        for rname, fn in active_readouts():
            top_t, bot_t = fn(vec)
            print(f"    {label:<10} |cos|={ac:.3f}  {rname:<8} + {' '.join(top_t[:N_TOP_TOKENS])}")
            if SHOW_BOTTOM:
                print(f"    {label:<10}              {rname:<8} - {' '.join(bot_t[:N_TOP_TOKENS])}")

In [ ]:
toks = ["breezy", "tornado", "hurricane", "thunder"]
sweep_layers = [20, 21, 22]

for LAYER in sweep_layers:
    s   = compute_support(toks)
    ap  = predicted_displacement(s["prototype"], LAYER)
    svs, tbl = sv_directions(s["prototype"], LAYER)

    print(f"\n{'=' * 110}\n{' '.join(toks).upper()}   layer {LAYER}   "
          f"cos(A@p, d) = {cos(ap, s['disp'][LAYER]):+.3f}\n{'=' * 110}")
    print_directions({"p": s["prototype"], "d": s["disp"][LAYER], "A@p": ap})
    print("")
    print(f"aligned axes (ranked by {RANK_BY}): {list(tbl['sv'].astype(int))}")
    abs_cos_map = dict(zip(tbl["sv"].astype(int), tbl["abs_cos"]))
    for label, vec in svs.items():
        sv_i = int(label.split("SV")[1].split()[0])
        ac   = abs_cos_map[sv_i]
        for rname, fn in active_readouts():
            top_t, bot_t = fn(vec)
            print(f"    {label:<10} |cos|={ac:.3f}  {rname:<8} + {' '.join(top_t[:N_TOP_TOKENS])}")
            if SHOW_BOTTOM:
                print(f"    {label:<10}              {rname:<8} - {' '.join(bot_t[:N_TOP_TOKENS])}")

### 5b. Sweep categories 

Same layer, the category varying 

In [ ]:
SWEEP_CATEGORIES = ["sport", "disease", "drug"]
LAYER = 20

for cat in SWEEP_CATEGORIES:
    toks, prov = pick_support_set(cat, None, SUPPORT_PICK)
    s   = compute_support(toks)
    ap  = predicted_displacement(s["prototype"], LAYER)
    svs, tbl = sv_directions(s["prototype"], LAYER)

    print(f"\n{'=' * 110}\n{cat.upper()}  {toks}   layer {LAYER}   "
          f"cos(A@p, d) = {cos(ap, s['disp'][LAYER]):+.3f}\n{'=' * 110}")
    print_directions({"p": s["prototype"], "d": s["disp"][LAYER], "A@p": ap})
    print("")
    print(f"aligned axes (ranked by {RANK_BY}): {list(tbl['sv'].astype(int))}")
    abs_cos_map = dict(zip(tbl["sv"].astype(int), tbl["abs_cos"]))
    for label, vec in svs.items():
        sv_i = int(label.split("SV")[1].split()[0])
        ac   = abs_cos_map[sv_i]
        for rname, fn in active_readouts():
            top_t, bot_t = fn(vec)
            print(f"    {label:<10} |cos|={ac:.3f}  {rname:<8} + {' '.join(top_t[:N_TOP_TOKENS])}")
            if SHOW_BOTTOM:
                print(f"    {label:<10}              {rname:<8} - {' '.join(bot_t[:N_TOP_TOKENS])}")

### 5c. Sweep support sets 

In [24]:
RANK_BY       = "abs_cos"     # "gain" | "abs_cos" | "energy" | "sigma" for SVs
support_sets = [["tomato", "rose", "cherry", "cardinal"]]
LAYER = 23

for toks in support_sets:
    #s   = compute_support(toks)
    tok_ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in toks]
    prototype = EMB_T[tok_ids].mean(0).cpu().numpy()  # no forward pass
    svs, tbl = sv_directions(prototype, LAYER)
    #svs, tbl = sv_directions(s["prototype"], LAYER)

    print(f"\n{'=' * 110}\n{' '.join(toks).upper()}   layer {LAYER}   ")
    print_directions({"p": s["prototype"], "d": s["disp"][LAYER]})
    print("")
    print(f"aligned axes (ranked by {RANK_BY}): {list(tbl['sv'].astype(int))}")
    abs_cos_map = dict(zip(tbl["sv"].astype(int), tbl["abs_cos"]))
    for label, vec in svs.items():
        sv_i = int(label.split("SV")[1].split()[0])
        ac   = abs_cos_map[sv_i]
        for rname, fn in active_readouts():
            top_t, bot_t = fn(vec)
            print(f"    {label:<10} |cos|={ac:.3f}  {rname:<8} + {' '.join(top_t[:N_TOP_TOKENS])}")
            if SHOW_BOTTOM:
                print(f"    {label:<10}              {rname:<8} - {' '.join(bot_t[:N_TOP_TOKENS])}")


TOMATO ROSE CHERRY CARDINAL   layer 23   

p
    cos      ▁cherry ▁tomato ▁cardinal ▁Tomato tomato cherry ▁ROSE Tomato ▁tomatoes ▁cherries

d
    cos      ▁blue ▁crimson blue ▁pink ▁rose ▁yellow ▁purple pink ▁cherry cherry

aligned axes (ranked by abs_cos): [29, 4, 31, 33, 11]
    SV29 in    |cos|=0.099  cos      + ▁purple ▁red ▁pink ▁yellow ▁rose red ▁rosas purple ▁RED yellow
    SV29 in                 cos      - ▁animal ▁bras ▁Falcon ▁attic ▁testosterone ▁clothing ▁animals ▁canine ▁python ▁human
    SV29 out   |cos|=0.099  cos      + ▁blue ▁green blue ▁available ▁purple ▁elegantly ▁artisan の ▁candy green
    SV29 out                cos      - predator ⋞ 🪓 crocs BASS ichester ▁Zoll ZONE Predator 饃
    SV4 in     |cos|=0.085  cos      + ▁lemon ▁chicken ▁olive ▁trout 狗 ▁tomato olive ▁sheep ▁goat ▁orange
    SV4 in                  cos      - ▁anchor ▁anchors ▁drill ▁shovel ▁drills ▁door ▁nails ▁shears ▁shovels ▁scissors
    SV4 out    |cos|=0.085  cos      + ▁divertidos ▁divertido ▁or

In [26]:
# ── Feature-space projection of a prototype ──────────────────────────────────
# The ridge map decomposes as A = U S Vt. Applying it to prototype p gives:
#   p @ A = sum_i coord_i * v_i,   coord_i = sigma_i * (p · u_i)
#
# coord_i is the weight p places on output direction v_i — the actual coordinate
# of the predicted displacement in the SV basis. This differs from abs_cos:
#   abs_cos = |p_hat · u_i_hat|  normalises both vectors, loses magnitude info
#   coord_i = sigma_i * (p · u_i) keeps the full scale of the prototype and the gain

LAYER  = 23
toks   = ["tomato", "rose", "cherry", "cardinal", "heart"]
K_SHOW = 10   # how many top feature-space axes to display

tok_ids   = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in toks]
prototype = EMB_T[tok_ids].mean(0).cpu().numpy()

in_svecs, sigma, out_svecs = layer_svd(LAYER)   # (k, d), (k,), (k, d)

dots   = in_svecs @ prototype      # (k,)  raw dot products  p · u_i
coords = sigma * dots              # (k,)  feature-space coordinates

order = np.argsort(np.abs(coords))[::-1][:K_SHOW]

print(f"\n{'=' * 120}")
print(f"{' '.join(toks).upper()}   layer {LAYER}   ||p|| = {np.linalg.norm(prototype):.2f}")
print(f"Feature-space coordinates: coord_i = sigma_i * (p · u_i)")
print(f"{'=' * 120}")
print(f"{'SV':>4}  {'coord':>9}  {'sigma':>8}  {'p·u_i':>9}    input axis tokens (+/−)")
print(f"{'-' * 120}")
for i in order:
    top_t, bot_t = cosine_readout(in_svecs[i], n=8)
    print(f"  {i:>4}  {coords[i]:>+9.2f}  {sigma[i]:>8.2f}  {dots[i]:>+9.4f}"
          f"    + {' '.join(top_t)}")
    print(f"  {'':>4}  {'':>9}  {'':>8}  {'':>9}    - {' '.join(bot_t)}")


TOMATO ROSE CHERRY CARDINAL HEART   layer 23   ||p|| = 0.93
Feature-space coordinates: coord_i = sigma_i * (p · u_i)
  SV      coord     sigma      p·u_i    input axis tokens (+/−)
------------------------------------------------------------------------------------------------------------------------
     4      -0.67      9.06    -0.0738    + ▁anchor ▁anchors ▁drill ▁shovel ▁drills ▁door ▁nails ▁shears
                                          - ▁lemon ▁chicken ▁olive ▁trout 狗 ▁tomato olive ▁sheep
    29      +0.48      5.97    +0.0808    + ▁purple ▁red ▁pink ▁yellow ▁rose red ▁rosas purple
                                          - ▁animal ▁bras ▁Falcon ▁attic ▁testosterone ▁clothing ▁animals ▁canine
    31      -0.44      5.94    -0.0746    + ▁moths ▁geodes ▁ant ▁worms ▁Moth ▁moth Moth ▁underwear
                                          - ▁novel ▁pine ▁kitchen ▁vehicle ▁temperature ▁Novel ▁oak ▁action
     2      +0.42     10.78    +0.0386    + ▁rug ▁watch ▁cabinet ▁ring watch ▁l

In [32]:
# ── Feature-space projection of a prototype ──────────────────────────────────
# The ridge map decomposes as A = U S Vt. Applying it to prototype p gives:
#   p @ A = sum_i coord_i * v_i,   coord_i = sigma_i * (p · u_i)
#
# coord_i is the weight p places on output direction v_i — the actual coordinate
# of the predicted displacement in the SV basis. This differs from abs_cos:
#   abs_cos = |p_hat · u_i_hat|  normalises both vectors, loses magnitude info
#   coord_i = sigma_i * (p · u_i) keeps the full scale of the prototype and the gain

LAYER  = 23
toks   = ["cinco", "dix", "nove", "ocho"]
K_SHOW = 10   # how many top feature-space axes to display

tok_ids   = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in toks]
prototype = EMB_T[tok_ids].mean(0).cpu().numpy()

in_svecs, sigma, out_svecs = layer_svd(LAYER)   # (k, d), (k,), (k, d)

dots   = in_svecs @ prototype      # (k,)  raw dot products  p · u_i
coords = sigma * dots              # (k,)  feature-space coordinates

order = np.argsort(np.abs(coords))[::-1][:K_SHOW]

print(f"\n{'=' * 120}")
print(f"{' '.join(toks).upper()}   layer {LAYER}   ||p|| = {np.linalg.norm(prototype):.2f}")
print(f"Feature-space coordinates: coord_i = sigma_i * (p · u_i)")
print(f"{'=' * 120}")
print(f"{'SV':>4}  {'coord':>9}  {'sigma':>8}  {'p·u_i':>9}    input axis tokens (+/−)")
print(f"{'-' * 120}")
for i in order:
    top_t, bot_t = cosine_readout(in_svecs[i], n=20)
    print(f"  {i:>4}  {coords[i]:>+9.2f}  {sigma[i]:>8.2f}  {dots[i]:>+9.4f}"
          f"    + {' '.join(top_t)}")
    print(f"  {'':>4}  {'':>9}  {'':>8}  {'':>9}    - {' '.join(bot_t)}")


CINCO DIX NOVE OCHO   layer 23   ||p|| = 0.98
Feature-space coordinates: coord_i = sigma_i * (p · u_i)
  SV      coord     sigma      p·u_i    input axis tokens (+/−)
------------------------------------------------------------------------------------------------------------------------
    55      +0.53      4.95    +0.1078    + ▁eleven ▁eight ▁seven ▁fifteen ▁nine ▁seventeen ▁square eleven ▁five ▁fourteen fif ▁four ▁twelve ▁rectangular ▁twenty ▁thirteen ▁Square ▁fifty ▁forty ▁seventh
                                          - ▁pyramid ▁torus ▁cider ▁Pyramid ▁waterfall ▁plateau Pyramid ▁Hulu ▁mixers ▁pine ▁plaster pyramid ▁Plaster ▁prism ▁Cider ▁valley ▁church fruit ▁pyramids Hulu
     1      +0.44     12.67    +0.0351    + ▁senior ▁SENIOR Senior forward フロント ▁hierdie ▁передний ▁starboard ▁junior ▁reducido senior junior ▁koruyucu ▁Senior SENIOR ▁arany ▁wortel convite ▁coriander frontend
                                          - ▁paintings ▁land ▁movies ▁table ▁horse ▁truck ▁plants